# SMS Spam Detection using SVM

### Overview
This notebook demonstrates how to build a spam classifier using a Support Vector Machine (SVM). The goal is to accurately distinguish between legitimate messages (ham) and unsolicited messages (spam).

### 1. The Dataset
We will be using the **SMS Spam Collection Dataset** from Kaggle. It contains a set of SMS messages in English, pre-labeled as either 'ham' or 'spam'[cite: 9]. You can find the dataset at: `https://www.kaggle.com/uciml/sms-spam-collection-dataset`.

### 2. Text Preprocessing
Machine learning algorithms like SVM cannot process raw text directly; they require numerical input. To solve this, we must convert our text sentences into a mathematical representation. 

We will use tools from `sklearn.feature_extraction.text` to achieve this:
*   **`CountVectorizer`**: A simpler approach that creates a matrix of raw word counts (how many times each word appears in a sentence).
*   **`TfidfVectorizer` (TF-IDF)**: A more advanced technique that not only counts words but also weighs their importance. It penalizes common words (like "the", "is") and highlights words that are unique to specific messages.

The API for these vectorizers follows a standard structure:
1. Initialize the object: `obj = TfidfVectorizer()`
2. Learn the vocabulary: `obj.fit(sentences)`
3. Convert text to numbers: `X = obj.transform(sentences)`
* (Note: Steps 2 and 3 are typically combined using `X = obj.fit_transform(sentences)` for the training data).*

In [1]:
from sklearn.svm import SVC
from datetime import datetime
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.model_selection import train_test_split
from wordcloud import WordCloud

# data from:
# https://www.kaggle.com/uciml/sms-spam-collection-dataset
# file contains some invalid chars

df = pd.read_csv('spam.csv', encoding='ISO-8859-1')

# drop unnecessary columns
df = df.drop(["Unnamed: 2", "Unnamed: 3", "Unnamed: 4"], axis=1)

# rename columns to something better
df.columns = ['labels', 'data']

# create binary labels
df['b_labels'] = df['labels'].map({'ham': 0, 'spam': 1})
Y = df['b_labels'].values

# try multiple ways of calculating features
tfidf = TfidfVectorizer(decode_error='ignore')
X = tfidf.fit_transform(df['data'])

# count_vectorizer = CountVectorizer(decode_error='ignore')
# X = count_vectorizer.fit_transform(df['data'])

# split the data
Xtrain, Xtest, Ytrain, Ytest = train_test_split(X, Y, test_size=0.33)

model = SVC(kernel='linear', C=2.)

t0 = datetime.now()
model.fit(Xtrain, Ytrain)
print("train duration:", datetime.now() - t0)
t0 = datetime.now()
print("train score:", model.score(Xtrain, Ytrain), "duration:", datetime.now() - t0)
t0 = datetime.now()
print("test score:", model.score(Xtest, Ytest), "duration:", datetime.now() - t0)

# visualize the data
def visualize(label):
    words = ''
    for msg in df[df['labels'] == label]['data']:
        msg = msg.lower()
        words += msg + ' '
    wordcloud = WordCloud(width=600, height=400).generate(words)
    plt.imshow(wordcloud)
    plt.axis('off')
    plt.title(label)
    plt.show()

visualize('spam')
visualize('ham')

# see what we're getting wrong
df['predictions'] = model.predict(X)

# things that should be spam
print("*** things that should be spam ***")
sneaky_spam = df[(df['predictions'] == 0) & (df['b_labels'] == 1)]['data']
for msg in sneaky_spam:
    print(msg)

# things that should not be spam
print("*** things that should not be spam ***")
not_actually_spam = df[(df['predictions'] == 1) & (df['b_labels'] == 0)]['data']
for msg in not_actually_spam:
    print(msg)

ImportError: DLL load failed while importing _path: An Application Control policy has blocked this file.